In [1]:
import os, sys, json, re, unicodedata, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from transformers import BitsAndBytesConfig
import bitsandbytes as bnb

c:\Users\nguye\anaconda3\envs\llamaenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Cho phép import từ thư mục gốc ---
if "__file__" in globals():
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
else:
    # Nếu đang ở Notebook → thêm đường dẫn gốc project thủ công
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from Config.SemanticConfig import *

In [ ]:
INPUT_XLSX = os.path.join(BASE_DIR, "Documents", "Confessions of HNMU.xlsx")
COL_NAME = "post_text"
OUTPUT_JSON = os.path.join(BASE_DIR, "Output", "Semantic.json")
MAX_ROWS = 10
VERBOSE = True


In [3]:
# --- Thư mục lưu mô hình ---
MODEL_DIR = r"D:\Model"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

In [4]:
# =====================================================
# 🔹 1. CHUẨN HÓA VĂN BẢN
# =====================================================
def normalize_text(text):
    """Chuẩn hóa văn bản tiếng Việt trước khi đưa vào model."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = re.sub(r"[^a-zA-ZÀ-ỹ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
# =====================================================
# 🔹 2. TẢI MÔ HÌNH LLAMA-3.2-3B-INSTRUCT
# =====================================================
def load_llama_model():
    """Tự động tải hoặc load model Llama-3.2-3B-Instruct từ D:\Model."""
    try:
        print(f"📦 Kiểm tra mô hình trong cache: {MODEL_DIR} ...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=MODEL_DIR)
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,          # dùng 8-bit để giảm VRAM
            llm_int8_threshold=6.0,
            llm_int8_skip_modules=None,
        )

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            cache_dir=MODEL_DIR,
            quantization_config=bnb_config,
            device_map="cuda",           # ép chạy trên GPU
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        print(f"GPU Memory Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")        
        print(bnb.__version__)
        print(torch.cuda.is_available(), torch.version.cuda)
        print(f"✅ Model đã sẵn sàng tại cache: {MODEL_DIR}")
    except Exception as e:
        print(f"⚠️ Không thể load từ cache ({e}). Đang tải từ Internet...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
        model.save_pretrained(os.path.join(MODEL_DIR, "Llama-3.2-3B-Instruct"))
        tokenizer.save_pretrained(os.path.join(MODEL_DIR, "Llama-3.2-3B-Instruct"))
    return tokenizer, model


In [6]:
# =====================================================
# 🔹 3. PHÂN TÍCH NGỮ NGHĨA MỘT CÂU
# =====================================================
def classify_llama(text, generator):
    """Phân tích ngữ nghĩa văn bản bằng Llama-3.2-3B-Instruct."""
    text = normalize_text(text)
    if not text.strip():
        return "CLEAN", "Không có nội dung."

    prompt = f"""
    Hãy phân loại đoạn văn sau thành một trong ba nhãn:
    CLEAN (bình thường, không xúc phạm),
    OFFENSIVE (có ngôn ngữ xúc phạm hoặc thô tục),
    HATE (ngôn ngữ thù ghét hoặc tấn công cá nhân/nhóm).

    Trả về JSON ngắn gọn với 2 trường:
    {{
      "label": "CLEAN|OFFENSIVE|HATE",
      "reason": "giải thích ngắn vì sao"
    }}

    Đoạn văn: "{text}"
    """

    output = generator(prompt, max_new_tokens=200, temperature=0.1, do_sample=False)
    response = output[0]["generated_text"]

    # Cố gắng parse JSON trong output
    try:
        json_start = response.find("{")
        json_end = response.rfind("}") + 1
        parsed = json.loads(response[json_start:json_end])
        label = parsed.get("label", "CLEAN").upper()
        reason = parsed.get("reason", "").strip()
    except Exception:
        label, reason = "CLEAN", response.strip()

    return label, reason

In [ ]:
# =====================================================
# 🔹 4. PHÂN TÍCH FILE EXCEL
# =====================================================
def semantic_analysis_excel(excel_path, col_name):
    """Đọc file Excel, phân tích từng dòng bằng Llama-3.2-3B-Instruct."""
    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file Excel: {excel_path}")

    df = pd.read_excel(excel_path, nrows=MAX_ROWS)
    if col_name not in df.columns:
        raise ValueError(f"❌ Cột '{col_name}' không tồn tại trong file Excel.")

    texts = df[col_name].astype(str).fillna("").tolist()

    if VERBOSE:
        print(f"📘 Đã tải {len(texts)} dòng từ: {excel_path}")
        print(f"📑 Cột đang xử lý: {col_name}")
        print("🔹 Đang tải mô hình Llama-3.2-3B-Instruct...")

    tokenizer, model = load_llama_model()
    generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

    results = []
    for i, text in enumerate(texts, start=1):
        label, reason = classify_llama(text, generator)
        results.append({
            "index": i,
            col_name: text,
            "semantic_label": label,
            "explanation": reason
        })

        if VERBOSE and i % 10 == 0:
            print(f"  🔹 Đã xử lý {i}/{len(texts)} dòng...")

    return results

: 

In [ ]:
# =====================================================
# 🔹 5. CHẠY THỬ VÀ XUẤT JSON
# =====================================================
if __name__ == "__main__":
    print("🔍 Đang phân tích ngữ nghĩa bằng Llama-3.2-3B-Instruct...")

    data = semantic_analysis_excel(INPUT_XLSX, COL_NAME)

    total = len(data)
    hate = sum(1 for x in data if x["semantic_label"] == "HATE")
    offensive = sum(1 for x in data if x["semantic_label"] == "OFFENSIVE")
    clean = total - hate - offensive

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Phân tích hoàn tất {total} dòng.")
    print(f"💾 Kết quả lưu tại: {OUTPUT_JSON}")
    print(f"📊 CLEAN={clean}, OFFENSIVE={offensive}, HATE={hate}")


🔍 Đang phân tích ngữ nghĩa bằng Llama-3.2-3B-Instruct...
📘 Đã tải 10 dòng từ: e:\Personal\Group Assignments\Lap_trinh_web\ToxicFilter\Documents\Confessions of HNMU.xlsx
📑 Cột đang xử lý: post_text
🔹 Đang tải mô hình Llama-3.2-3B-Instruct...
📦 Kiểm tra mô hình trong cache: D:\Model ...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.35s/it]


⚠️ Không thể load từ cache (Calling `to()` is not supported for `4-bit` quantized models with the installed version of bitsandbytes. The current device is `cuda:0`. If you intended to move the model, please install bitsandbytes >= 0.43.2.). Đang tải từ Internet...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\nguye\anaconda3\envs\llamaenv\lib\site-packages\huggingface_hub\file_download.py:801: UserWarning: Not enough free disk space to download the file. The expected file size is: 4965.80 MB. The target location C:\Users\nguye\.cache\huggingface\hub\models--meta-llama--Llama-3.2-3B-Instruct\blobs only has 3135.91 MB free disk space.
  warnings.warn(
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]